# ModelDock — Getting Started (Ollama · LM Studio · llama.cpp)

**ModelDock** is a lightweight, Python-first *model manager* for local LLMs.
It discovers, downloads, caches, verifies, and loads local models through
pluggable runtime adapters — and then hands you the **runtime's own client**
so you can run inference directly.

This notebook shows you how to use the `modeldock` SDK (v0.2.0) to talk to
three backends:

| Backend | Local server | Client ModelDock returns | Talks via |
|---|---|---|---|
| **Ollama** | `ollama serve` (`:11434`) | `ollama.Client` | Ollama SDK |
| **LM Studio** | LM Studio app (`:1234`) | `openai.OpenAI` | OpenAI-compatible API |
| **llama.cpp** | `llama-server` (`:8080`) | `openai.OpenAI` | OpenAI-compatible API |

> **The one idea to remember:** ModelDock *manages* models; it does **not** run
> inference. `mgr.load(name)` returns the backend's native client, and the API
> you call next depends on which backend that is. This notebook shows both the
> raw per-backend usage **and** a small `ModelDockChat` helper that hides the
> difference.

Every section is **guarded**: if a backend's server isn't running, the cell
prints setup instructions instead of erroring — so you can open this notebook
anywhere and run the parts you have installed.

## The mental model

```
You ──► modeldock (discover / install / cache / verify) ──► runtime-native client
                                                              │
                              Ollama: ollama.Client.chat / .generate
                              LM Studio / llama.cpp: openai.OpenAI.chat.completions.create
```

`md.Manager(backend="ollama" | "lmstudio" | "llamacpp")` scopes every operation
to one runtime. `mgr.load(name)` resolves the model, installs it if missing
(when the backend supports that), verifies it, and returns a ready client.

## ⚙️ Configuration — edit this first

Set the model name (and host, if non-default) for each backend you have
installed. The rest of the notebook reads from this `CONFIG` dict, so you only
edit model names in one place.

In [ ]:
CONFIG = {
    # Ollama: a tag from `ollama list` / `modeldock search`
    "ollama":   {"model": "llama3", "host": "http://localhost:11434"},

    # LM Studio: a Hugging Face coordinate (the model's "Identifier" in LM Studio)
    "lmstudio": {"model": "lmstudio-community/Qwen2.5-Coder-7B-Instruct-GGUF",
                 "host": "http://localhost:1234"},

    # llama.cpp: the GGUF filename your `llama-server` loaded
    "llamacpp": {"model": "your-model.gguf", "host": "http://localhost:8080",
                 "gpu_layers": -1},
}
print("Edit CONFIG above to match the models you have loaded in each backend.")

## §0 Install

Install ModelDock (with the Ollama SDK extra) and the **OpenAI SDK**.

> ⚠️ The OpenAI SDK is **not** a declared dependency of ModelDock, but the
> LM Studio and llama.cpp adapters use it to talk to their servers. Install it
> explicitly. If you already have these packages, the cell is a no-op.

In [ ]:
import subprocess, sys

def ensure(pkg):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        print(f"installed {pkg}")
    except Exception as e:
        print(f"Could not install {pkg} (install manually): {e}")

ensure("modeldock[ollama]")
ensure("openai")

In [ ]:
import modeldock as md
import ollama
from openai import OpenAI

print("modeldock", md.__version__)

## §1 How ModelDock connects to each backend

ModelDock's `RuntimeRegistry` maps a backend name to an adapter. Each adapter
connects over HTTP to that backend's local server and returns its native client:

```
OllamaRuntime   ──HTTP──► http://localhost:11434 ──► ollama.Client
LMStudioRuntime ──HTTP──► http://localhost:1234 ──► openai.OpenAI(base_url=.../v1)
LlamaCppRuntime ──HTTP──► http://localhost:8080 ──► openai.OpenAI(base_url=.../v1)
```

**Hosts** are resolved from (in order): an explicit argument → environment
variable → conventional default.

| Backend | Env var | Default |
|---|---|---|
| Ollama | `MODELDOCK_OLLAMA_HOST` | `http://localhost:11434` |
| LM Studio | `MODELDOCK_LMSTUDIO_HOST` | `http://localhost:1234` |
| llama.cpp | `MODELDOCK_LLAMACPP_HOST` | `http://localhost:8080` |
| llama.cpp | `MODELDOCK_LLAMACPP_GPU_LAYERS` | (none → CPU) |

The helper below reads `CONFIG`, sets the right env var, and returns a
`ModelManager` scoped to that backend. `backend_ready()` probes whether the
server is reachable so later cells can skip gracefully.

In [ ]:
import os

def get_manager(backend):
    """Build a ModelManager for `backend`, applying the host from CONFIG."""
    cfg = CONFIG.get(backend, {})
    host = cfg.get("host")
    if backend == "ollama" and host:
        os.environ["MODELDOCK_OLLAMA_HOST"] = host
    elif backend == "lmstudio" and host:
        os.environ["MODELDOCK_LMSTUDIO_HOST"] = host
    elif backend == "llamacpp" and host:
        os.environ["MODELDOCK_LLAMACPP_HOST"] = host
    if backend == "llamacpp" and "gpu_layers" in cfg:
        os.environ["MODELDOCK_LLAMACPP_GPU_LAYERS"] = str(cfg["gpu_layers"])
    return md.Manager(backend=backend)

def backend_ready(backend):
    """Return (available: bool, status) and print a setup hint if down."""
    try:
        mgr = get_manager(backend)
        status = mgr.runtime_status()   # RuntimeStatus(available, device, ...)
        if status.available:
            print(f"[OK] {backend} reachable ({status.device.value})")
            return True, status
        print(f"[!] {backend} not available: {status.details or 'server not reachable'}")
        return False, status
    except Exception as e:
        print(f"[!] {backend} check failed: {e}")
        return False, None

## §2 Discovery (backend-agnostic)

These operations work the same regardless of backend. Pass `backend=` to scope
discovery to a runtime's own naming (e.g. LM Studio returns Hugging Face
coordinates, not Ollama tags).

In [ ]:
# What's in the catalog?
print("categories:", [c.value for c in md.categories()][:8], "...")

# Search by name / capability / category
hits = md.search("coding")
print("search('coding'):", [getattr(h, "name", h) for h in hits][:5])

# What's already installed in the *default* backend?
print("installed:", [m.qualified_name() for m in md.installed()])

# Details + resolve a friendly name to its canonical spec
info = md.info("llama3")
print("info(llama3):", type(info).__name__)
spec = md.resolve("qwen3")
print("resolve(qwen3) source:", getattr(spec, "source", None))

## §3 Backend A — Ollama (raw usage)

Ollama is the only backend with **full programmatic install**: `mgr.install()`
pulls the model from ollama.com. `mgr.load()` then returns an `ollama.Client`.

Run `ollama serve` (or open the Ollama desktop app) before running this cell.

In [ ]:
backend = "ollama"
ready, _ = backend_ready(backend)
if ready:
    mgr = get_manager(backend)
    model = CONFIG[backend]["model"]

    # Install only if it isn't already local
    if not any(m.name == model for m in mgr.installed()):
        print(f"Installing {model} ...")
        mgr.install(model)

    client = mgr.load(model)          # -> ollama.Client
    print("Client type:", type(client).__name__)

    # --- raw Ollama chat ---
    resp = client.chat(model=model,
                       messages=[{"role": "user", "content": "Say hello in one word."}])
    print("chat ->", resp["message"]["content"])

    # --- raw Ollama streaming generate ---
    print("stream ->", end=" ")
    for ev in client.generate(model=model, prompt="Count to 3.", stream=True):
        print(ev["response"], end="", flush=True)
    print()

## §4 Backend B — LM Studio (raw usage)

LM Studio exposes an **OpenAI-compatible** server. `mgr.load()` returns an
`openai.OpenAI` client pointed at `http://localhost:1234/v1`.

1. Open LM Studio → **Local Server** → load a model → **Start Server**.
2. Use the model's **Identifier** (a Hugging Face coordinate) in `CONFIG`.

> Some models must be downloaded through the LM Studio UI; `mgr.install()`
> triggers a download call but may point you to the app for certain models.

In [ ]:
backend = "lmstudio"
ready, _ = backend_ready(backend)
if ready:
    mgr = get_manager(backend)
    model = CONFIG[backend]["model"]

    client = mgr.load(model)          # -> openai.OpenAI(base_url=.../v1)
    print("Client type:", type(client).__name__)

    # --- raw OpenAI-compatible chat completion ---
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": "Say hello in one word."}],
    )
    print("chat ->", resp.choices[0].message.content)

    # --- streaming ---
    print("stream ->", end=" ")
    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": "Count to 3."}],
        stream=True,
    )
    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            print(delta, end="", flush=True)
    print()

## §5 Backend C — llama.cpp (raw usage)

llama.cpp also exposes an **OpenAI-compatible** server, so the client API is
identical to LM Studio. `mgr.load()` returns `openai.OpenAI` at `:8080/v1`.

> ⚠️ **llama.cpp has no programmatic install.** You must start the server
> yourself with the GGUF you want:
>
> ```bash
> llama-server -m path/to/your-model.gguf --host 0.0.0.0 --port 8080
> # GPU: add --gpu-layers 35  (or set MODELDOCK_LLAMACPP_GPU_LAYERS in CONFIG)
> ```
>
> `model` in `CONFIG` is the **filename** the server reports (e.g.
> `your-model.gguf`). `mgr.install()` / `mgr.update()` intentionally raise an
> actionable error here.

In [ ]:
backend = "llamacpp"
ready, _ = backend_ready(backend)
if ready:
    mgr = get_manager(backend)
    model = CONFIG[backend]["model"]

    client = mgr.load(model)          # -> openai.OpenAI(base_url=.../v1)
    print("Client type:", type(client).__name__)

    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": "Say hello in one word."}],
    )
    print("chat ->", resp.choices[0].message.content)

    # Streaming works exactly like LM Studio
    print("stream ->", end=" ")
    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": "Count to 3."}],
        stream=True,
    )
    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            print(delta, end="", flush=True)
    print()

## §6 Unified helper — `ModelDockChat`

The three backends return **different client types**, so the inference call
differs. This small wrapper hides that: it loads the model through ModelDock and
exposes `.chat()`, `.complete()`, and `.stream()` that work for **any** backend.

Internally it branches on `isinstance(client, ollama.Client)` — Ollama uses the
Ollama SDK, everything else uses the OpenAI SDK.

In [ ]:
class ModelDockChat:
    """Backend-agnostic chat wrapper around the client ModelDock returns."""

    def __init__(self, backend, model=None, auto_install=True):
        self.backend = backend
        self.model = model or CONFIG[backend]["model"]
        self.mgr = get_manager(backend)
        self.client = self.mgr.load(self.model, auto_install=auto_install)

    def chat(self, messages):
        if isinstance(self.client, ollama.Client):
            return self.client.chat(model=self.model, messages=messages)["message"]["content"]
        # OpenAI-compatible (LM Studio, llama.cpp)
        return self.client.chat.completions.create(
            model=self.model, messages=messages
        ).choices[0].message.content

    def complete(self, prompt):
        return self.chat([{"role": "user", "content": prompt}])

    def stream(self, prompt):
        if isinstance(self.client, ollama.Client):
            for ev in self.client.generate(model=self.model, prompt=prompt, stream=True):
                yield ev["response"]
        else:
            stream = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                stream=True,
            )
            for chunk in stream:
                delta = chunk.choices[0].delta.content
                if delta:
                    yield delta

### Demo: one backend-agnostic loop

The same three lines run on **whichever backends are currently up** — no
per-backend branching in your own code.

In [ ]:
for backend in ["ollama", "lmstudio", "llamacpp"]:
    ready, _ = backend_ready(backend)
    if not ready:
        continue
    try:
        bot = ModelDockChat(backend)
        print(f"\n=== {backend} ({bot.model}) ===")
        print("answer:", bot.complete("What is 2+2? Reply with just the number."))
        print("stream:", end=" ")
        for tok in bot.stream("List 2 fruits."):
            print(tok, end="", flush=True)
        print()
    except Exception as e:
        print(f"{backend} demo failed: {e}")

## §7 Lifecycle & operations

ModelDock's management features (install / update / remove / verify / cache) are
backend-scoped. `md.run()` returns a `RunResult` (`.success`, `.completion_tokens`,
`.error`) — **not** text — which is why you use the native client for output.

In [ ]:
backend = "ollama"
ready, _ = backend_ready(backend)
if ready:
    mgr = get_manager(backend)
    name = CONFIG[backend]["model"]

    print("installed:", [m.qualified_name() for m in mgr.installed()])
    print("verified :", mgr.verify(name))

    print("cache status:")
    for row in mgr.cache.status():
        print("  ", row)

    print("cache path:", mgr.cache.path())

    # Destructive / mutating ops — uncomment to use:
    # mgr.update(name, confirm=True)   # removes + re-pulls
    # mgr.remove(name)                 # uninstall
    # mgr.cache.clean()                # no args in v0.2.0

## §8 FAQ / Troubleshooting

**`[!] ollama not available`** — start the server: `ollama serve` (or open the
desktop app). Check `MODELDOCK_OLLAMA_HOST` if it's on another host/port.

**`[!] lmstudio not available`** — In LM Studio, go to **Local Server**, load a
model, and click **Start Server**. Default port is `1234`.

**`[!] llamacpp not available`** — You must launch the server yourself:
`llama-server -m model.gguf --port 8080`. ModelDock cannot start it for you.

**`ModuleNotFoundError: openai`** — LM Studio and llama.cpp need the OpenAI SDK,
which is *not* a ModelDock dependency. Run `pip install openai`.

**`mgr.install()` fails for llama.cpp** — Expected. llama.cpp has no download API;
load the GGUF via `llama-server` and reference it by filename.

**`md.run()` returned something without text** — `run()` returns a `RunResult`
object (success/token counts), not the generated string. Use the native client
(`client.chat.completions.create(...)` or `client.generate(...)`) when you need
the text, as shown above.

**Model name formats differ per backend** — Ollama uses tags (`llama3`),
LM Studio uses Hugging Face coordinates (`lmstudio-community/...-GGGUF`),
llama.cpp uses the server's loaded GGUF filename. Set each in `CONFIG`.

**Docs vs code** — `CONTEXT.md`/`QUICKSTART.md` say only Ollama ships; in v0.2.0
Ollama, LM Studio, and llama.cpp are all implemented. This notebook reflects the
actual code.